# Phase 2 — Pretraining Model H (Hindi) and Model L (Nepali)

Decoder-only Transformers built from primitive PyTorch layers. Two fully independent
models: separate corpora, tokenizers, vocabularies and weights.

## How to use this notebook

1. **Settings -> Accelerator -> GPU T4 x2** before anything else.
2. Set `LANGUAGE` in the next cell to `"hi"` or `"ne"`.
3. Run all cells.

**If the session dies, just run all cells again with the same `LANGUAGE`.** Training
resumes from the last checkpoint at the exact step, optimiser state and batch order --
nothing is repeated and nothing is lost. Kaggle terminates sessions at 12 hours, so
`MAX_MINUTES` stops cleanly before that and writes a final checkpoint.

Run the notebook once with `LANGUAGE = "hi"`, then again with `"ne"`.


## 1. Control panel

Everything you would normally want to change lives here.


In [ ]:
LANGUAGE      = "hi"      # "hi" = Hindi (Model H), "ne" = Nepali (Model L)

RUN_PROBE     = True      # 50-step throughput measurement before committing to a full run
RUN_TRAINING  = True      # the pretraining run itself
RUN_EVAL      = True      # perplexity / BPB / generation / attention, after training

MAX_MINUTES   = 660       # stop and checkpoint before Kaggle's 12-hour session limit
MAX_STEPS     = None      # None = use the value in train.json (16,000). Lower it if the
                          # probe shows throughput too low to finish in time.
MICRO_BATCH   = None      # None = use train.json (32). Drop to 16 or 8 on a CUDA OOM;
                          # gradient accumulation compensates, so the effective batch
                          # and the learning dynamics are unchanged.
AMP_DTYPE     = None      # None = train.json ("float16"). Set "float32" if loss goes nan.

EVAL_PROMPTS  = 200       # held-out prompts for the generation metrics

assert LANGUAGE in ("hi", "ne")
LANG_DIR = {"hi": "hindi", "ne": "nepali"}[LANGUAGE]
MODEL_LABEL = {"hi": "Model H (Hindi, higher-resource)",
               "ne": "Model L (Nepali, lower-resource)"}[LANGUAGE]
print(f"configured for {MODEL_LABEL}")


## 2. Environment

Confirms the GPU is actually attached. Without one, training is a non-starter.


In [ ]:
import os, sys, json, time, shutil, subprocess
from pathlib import Path
import torch

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  {props.name}  {props.total_memory/1e9:.1f} GB  "
          f"| bf16 supported: {torch.cuda.is_bf16_supported()}")
    print(f"  visible devices: {torch.cuda.device_count()}")
else:
    print("\n  !! NO GPU ATTACHED !!")
    print("  Settings -> Accelerator -> GPU T4 x2, then re-run.")

print("\nworking dir space:")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)


## 3. Assemble the project tree

Kaggle mounts each uploaded folder as a separate read-only dataset and flattens the
nesting, so the scripts cannot run against `/kaggle/input` directly -- they expect
`hindi/`, `nepali/`, `lma/`, `scripts/` and `common/` in one writable place.

Data files are **symlinked** (no copy, no disk cost, instant). Code is **copied**, since
it is under a megabyte and Python writes bytecode caches next to modules.


In [ ]:
INPUT = Path("/kaggle/input")
ROOT  = Path("/kaggle/working/vidhi")


def link(src: Path, dst: Path) -> None:
    """Symlink one input file into the working tree, replacing any previous link."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src)


def for_language(pattern: str, lang: str) -> list:
    """Find input files matching a pattern whose path names the given language."""
    stem = lang[:5].lower()
    return sorted(p for p in INPUT.rglob(pattern) if stem in str(p).lower())


missing = []
for lang, code_ in (("hindi", "hi"), ("nepali", "ne")):
    for split in ("train", "validation", "test"):
        for ext in ("bin", "meta.json"):
            hits = for_language(f"{split}.{ext}", lang)
            if hits: link(hits[0], ROOT/lang/"data"/"tokens"/f"{split}.{ext}")
            else:    missing.append(f"{lang}/data/tokens/{split}.{ext}")

    hits = list(INPUT.rglob(f"{code_}.model"))
    if hits: link(hits[0], ROOT/lang/"tokenizer"/f"{code_}.model")
    else:    missing.append(f"{lang}/tokenizer/{code_}.model")

    for cfg in ("dataset.json", "model.json", "train.json"):
        hits = for_language(cfg, lang)
        if hits: link(hits[0], ROOT/lang/"configs"/cfg)
        else:    missing.append(f"{lang}/configs/{cfg}")

    shards = sorted(INPUT.rglob(f"{code_}-test-*.jsonl.zst"))
    for shard in shards:
        link(shard, ROOT/lang/"data"/"splits"/"test"/shard.name)
    if not shards:
        missing.append(f"{lang}/data/splits/test/*.jsonl.zst")

# Code datasets -> package directories. Kept as three separate uploads precisely so
# scripts/clean.py and common/clean.py stay distinguishable.
PACKAGES = {"automation-scripts": "scripts", "common": "common",
            "language-models-agents": "lma"}
for slug, package in PACKAGES.items():
    sources = [p for p in INPUT.rglob(slug) if p.is_dir()]
    if not sources:
        missing.append(f"code dataset {slug!r}")
        continue
    (ROOT/package).mkdir(parents=True, exist_ok=True)
    for f in sources[0].glob("*.py"):
        shutil.copy2(f, ROOT/package/f.name)
    n_modules = len(list((ROOT/package).glob("*.py")))
    print(f"  {package}/  {n_modules:>2} modules")

if missing:
    raise SystemExit("MISSING INPUTS:\n  " + "\n  ".join(missing))

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f"\nproject tree ready at {ROOT}")
train_bin = ROOT/LANG_DIR/"data"/"tokens"/"train.bin"
print(f"train corpus: {train_bin.stat().st_size/1e9:.2f} GB")


## 4. Dependencies


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentencepiece", "sacrebleu", "zstandard"], check=True)
import sentencepiece, sacrebleu, zstandard
print("sentencepiece", sentencepiece.__version__,
      "| sacrebleu", sacrebleu.__version__,
      "| zstandard", zstandard.__version__)

# A Devanagari-capable font. Without one, matplotlib draws every Hindi/Nepali glyph as
# an empty box, and the attention heatmaps lose the thing the spec asks them to show:
# the example sentence in the model's own language. Kaggle images do not reliably
# register one, so install it here and rebuild matplotlib's font cache.
from matplotlib import font_manager
import matplotlib

DEVANAGARI = ("Noto Sans Devanagari", "Noto Serif Devanagari", "Lohit Devanagari",
              "Nirmala UI", "Samyak Devanagari", "Kalimati", "FreeSerif")

def devanagari_font():
    """Return the first installed font that can render Devanagari, else None."""
    available = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in DEVANAGARI if n in available), None)

found = devanagari_font()
if found is None:
    print("no Devanagari font registered - installing fonts-indic ...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-indic", "fonts-noto-core"],
                   check=False, capture_output=True)
    # font_manager caches its list at import; force a rescan of the system font dirs.
    for path in font_manager.findSystemFonts(fontpaths=None, fontext="ttf"):
        try:
            font_manager.fontManager.addfont(path)
        except Exception:
            pass
    found = devanagari_font()

print("Devanagari font:", found or "NONE - heatmap axes will fall back to positions")


## 5. Sanity checks

The `tests/` folder is not uploaded, so the two checks that matter most are inlined here
and run against **this** PyTorch build and **this** GPU, not the laptop they were
originally verified on.

The causal-mask check is the important one. A masking bug is silent -- the model still
trains and the loss still falls, but it has been allowed to see the future and every
number downstream is meaningless. Equality is asserted **exactly**: a masked position's
softmax weight is `exp(-inf)`, which is precisely 0.0, so a future token must change
nothing at all in the past.


In [ ]:
from lma.config import ModelConfig, TrainConfig
from lma.data import TokenStream
from lma.model import GPT

cfg = ModelConfig.from_json(f"{LANG_DIR}/configs/model.json")
tcfg = TrainConfig.from_json(f"{LANG_DIR}/configs/train.json")
model = GPT(cfg)
counts = cfg.count_parameters()

assert model.num_parameters() == counts["total"], "analytic count != allocated count"
print(f"parameters      {counts['total']:,}  (target ~25,000,000)")
print(f"  embedding     {counts['token_embedding']:,} "
      f"({100*counts['token_embedding']/counts['total']:.1f}%)")
print(f"  non-embedding {counts['non_embedding']:,}")
print(f"  d_model {cfg.d_model} | layers {cfg.n_layer} | heads {cfg.n_head} "
      f"(d_head {cfg.d_head}) | ffn {cfg.d_ff} | context {cfg.max_seq_len}")

model.eval()
with torch.no_grad():
    ids = torch.randint(0, cfg.vocab_size, (2, 24))
    base, _, _ = model(ids)
    edited = ids.clone()
    edited[:, 12:] = (edited[:, 12:] + 1) % cfg.vocab_size
    perturbed, _, _ = model(edited)
    past   = (base[:, :12] - perturbed[:, :12]).abs().max().item()
    future = (base[:, 12:] - perturbed[:, 12:]).abs().max().item()

assert past == 0.0, f"CAUSAL MASK LEAKS: past logits moved by {past:.3e}"
assert future > 0.0, "model ignores its input entirely - check is vacuous"
print(f"\ncausal mask     past drift {past} (exact), future reacts by {future:.3f}")

for split in ("train", "validation", "test"):
    s = TokenStream(f"{LANG_DIR}/data/tokens/{split}.bin")
    assert s.meta["vocab_size"] == cfg.vocab_size, "corpus/config vocabulary mismatch"
    print(f"  {split:<11} {len(s):>14,} tokens  {s.meta['n_documents']:>9,} docs  "
          f"{s.utf8_bytes/1e9:.2f} GB utf-8")


## 6. Throughput probe

Fifty steps, to answer one question before committing hours: **how many tokens per second
does this GPU actually deliver?** That figure determines whether the full 16,000 steps
fits inside a session, and it cannot be known in advance -- the fp16 autocast and gradient
scaler paths have never run on CUDA before this cell.

Watch for `nan` in the loss. If it appears, set `AMP_DTYPE = "float32"` above and re-run.


In [ ]:
if RUN_PROBE:
    probe = json.loads(Path(f"{LANG_DIR}/configs/train.json").read_text())
    probe.update(max_steps=50, warmup_steps=10, eval_every=50,
                 eval_batches=10, checkpoint_every=10_000)
    if MICRO_BATCH: probe["micro_batch_size"] = MICRO_BATCH
    if AMP_DTYPE:   probe["amp_dtype"] = AMP_DTYPE
    Path("probe_train.json").write_text(json.dumps(probe, indent=2))

    started = time.time()
    subprocess.run([sys.executable, "-m", "scripts.pretrain", "--lang", LANGUAGE,
                    "--train-config", "probe_train.json",
                    "--out-dir", "/kaggle/working/probe",
                    "--log-every", "10"], check=True)

    log = Path("/kaggle/working/probe/train_log.jsonl")
    rates = [json.loads(l)["tokens_per_second"] for l in log.read_text().splitlines()
             if l.strip() and "tokens_per_second" in l]
    if rates:
        # Discard the first window: it carries CUDA context creation and kernel autotuning.
        steady = rates[1:] or rates
        rate = sum(steady) / len(steady)
        total = tcfg.max_steps * tcfg.tokens_per_step
        print(f"\n{'='*66}")
        print(f"  steady-state throughput   {rate/1e3:,.1f}k tokens/second")
        print(f"  full run  {tcfg.max_steps:,} steps x {tcfg.tokens_per_step:,} "
              f"= {total:,} tokens")
        print(f"  estimated wall clock      {total/rate/3600:.2f} hours")
        if total/rate/3600 > MAX_MINUTES/60:
            fits = int(rate * MAX_MINUTES * 60 / tcfg.tokens_per_step)
            print(f"\n  !! exceeds MAX_MINUTES. One session covers ~{fits:,} steps.")
            print(f"     Either re-run this notebook to resume, or set "
                  f"MAX_STEPS = {fits//1000*1000:,}.")
        else:
            print(f"  fits comfortably inside one {MAX_MINUTES/60:.1f}-hour session")
        print("="*66)
else:
    print("probe skipped")


## 7. Pretraining

AdamW, linear warmup into cosine decay, mixed precision, gradient accumulation to a fixed
token budget per step, and periodic validation.

`--resume` makes this cell idempotent: run it on a fresh session and it starts from zero;
run it after a kill and it continues from the last checkpoint with the same optimiser
moments, RNG state and batch order. Resume was verified bit-exact -- all 31 weight tensors
identical between an uninterrupted run and a resumed one.


In [ ]:
if RUN_TRAINING:
    train_cfg = json.loads(Path(f"{LANG_DIR}/configs/train.json").read_text())
    if MAX_STEPS:   train_cfg["max_steps"] = MAX_STEPS
    if MICRO_BATCH: train_cfg["micro_batch_size"] = MICRO_BATCH
    if AMP_DTYPE:   train_cfg["amp_dtype"] = AMP_DTYPE
    Path("run_train.json").write_text(json.dumps(train_cfg, indent=2))

    ckpt_dir = Path(f"/kaggle/working/checkpoints/{LANG_DIR}")
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    print(f"training {MODEL_LABEL}")
    print(f"checkpoints -> {ckpt_dir}\n")
    subprocess.run([sys.executable, "-m", "scripts.pretrain", "--lang", LANGUAGE,
                    "--train-config", "run_train.json",
                    "--out-dir", str(ckpt_dir),
                    "--max-minutes", str(MAX_MINUTES),
                    "--log-every", "50", "--resume"], check=True)
else:
    print("training skipped")


## 8. Evaluation

Runs only if a checkpoint exists. Four stages: intrinsic metrics, generation quality,
attention analysis, and loss curves.

If training stopped early because the session ran out, skip this, re-run the notebook to
resume training, and evaluate once `max_steps` is reached.


In [ ]:
ckpt_dir = Path(f"/kaggle/working/checkpoints/{LANG_DIR}")
best = ckpt_dir / "best.pt"

if RUN_EVAL and best.exists():
    payload = torch.load(best, map_location="cpu", weights_only=False)
    print(f"evaluating checkpoint at step {payload['step']:,} "
          f"({payload['tokens_seen']:,} tokens seen)\n")

    stages = [
        ["-m", "scripts.eval_lm", "--lang", LANGUAGE, "--checkpoint", str(best),
         "--splits", "validation", "test", "--batch-size", "16"],
        ["-m", "scripts.eval_generation", "--lang", LANGUAGE, "--checkpoint", str(best),
         "--num-prompts", str(EVAL_PROMPTS), "--prompt-tokens", "64",
         "--continuation-tokens", "128", "--batch-size", "16"],
        ["-m", "scripts.attention_analysis", "--lang", LANGUAGE, "--checkpoint", str(best),
         "--num-batches", "20", "--batch-size", "4", "--heads", "0", "1", "2", "3"],
        ["-m", "scripts.plot_training", "--lang", LANGUAGE,
         "--log", str(ckpt_dir / "train_log.jsonl")],
    ]
    for stage in stages:
        name = stage[1].split(".")[-1]
        print(f"\n{'-'*66}\n  {name}\n{'-'*66}")
        subprocess.run([sys.executable] + stage, check=False)
elif RUN_EVAL:
    print(f"no checkpoint at {best} - train first")
else:
    print("evaluation skipped")


## 9. Package the results

Bundles everything worth keeping into one archive under `/kaggle/working`, which appears
in the notebook's **Output** tab for download. Checkpoints go to Google Drive; the JSON
and figures go into `report/` on the phase-2 branch, because the assignment is explicit
that graders will not open Drive folders to find figures.


In [ ]:
import tarfile

stamp = f"{LANG_DIR}"
out = Path(f"/kaggle/working/phase2-{stamp}-results.tar.gz")
ckpt_dir = Path(f"/kaggle/working/checkpoints/{LANG_DIR}")

with tarfile.open(out, "w:gz") as tar:
    for pattern in ("best.pt", "step-*.pt", "train_log.jsonl",
                    "model_config.json", "train_config.json", "config.json"):
        for f in sorted(ckpt_dir.glob(pattern)):
            tar.add(f, arcname=f"checkpoints/{LANG_DIR}/{f.name}")
    report = ROOT / "report" / LANG_DIR
    if report.exists():
        for f in sorted(report.rglob("*")):
            if f.is_file():
                tar.add(f, arcname=f"report/{LANG_DIR}/{f.relative_to(report)}")

print(f"wrote {out}  ({out.stat().st_size/1e9:.2f} GB)\n")
print("contents:")
with tarfile.open(out) as tar:
    for m in tar.getmembers():
        print(f"  {m.size/1e6:9.2f} MB  {m.name}")

other = "ne" if LANGUAGE == "hi" else "hi"
print("\nNEXT: download from the Output tab, then run this notebook again with")
print(f'      LANGUAGE = "{other}"')
